In [361]:
import torch
from torchvision.models import (mobilenet_v3_small, MobileNet_V3_Small_Weights)

In [363]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

weights = MobileNet_V3_Small_Weights.DEFAULT

mobilenet = mobilenet_v3_small(weights=weights)
mobilenet.eval()
mobilenet.to(device)

preprocess = weights.transforms()

In [365]:
from PIL import Image

In [367]:
def extract_feature(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.inference_mode():
        feature = mobilenet.features(image_tensor)
        feature = mobilenet.avgpool(feature)
        feature = torch.flatten(feature, 1)

    return feature.squeeze(0).cpu()

In [369]:
feature = extract_feature("../data/labeled_data/img/C0002/after_1.png")

In [371]:
import torch
import torch.nn.functional as F

In [373]:
def build_event_feature(df):
    before_paths = [
        "../data/labeled_data" + df["before_1"],
        "../data/labeled_data" + df["before_2"],
        "../data/labeled_data" + df["before_3"]
    ]
    after_paths = [
        "../data/labeled_data" + df["after_1"],
        "../data/labeled_data" + df["after_2"],
        "../data/labeled_data" + df["after_3"]
    ]
    before_features = torch.stack([
        extract_feature(path) for path in before_paths
    ]).float()
    after_features = torch.stack([
        extract_feature(path) for path in after_paths
    ]).float()

    if torch.isnan(before_features).any():
        raise ValueError("before_features 中存在 NaN")
    if torch.isnan(after_features).any():
        raise ValueError("after_features 中存在 NaN")
    if torch.isinf(before_features).any():
        raise ValueError("before_features 中存在 Inf")
    if torch.isinf(after_features).any():
        raise ValueError("after_features 中存在 Inf")
        
    before_mean = before_features.mean(dim=0)
    after_mean = after_features.mean(dim=0)

    absolute_diff = torch.abs(before_mean - after_mean)

    similarities = []
    for before_feature in before_features:
        for after_feature in after_features:
            similarity = F.cosine_similarity(before_feature.unsqueeze(0),
                                            after_feature.unsqueeze(0), dim=1).item()
            similarities.append(similarity)

    if len(similarities) != 9:
        raise ValueError(f"相似度数量错误：{len(similarities)}")

    similarities_tensor = torch.tensor(
        similarities,
        dtype=torch.float32
    )

    if torch.isnan(similarities_tensor).any():
        raise ValueError("cosine similarities 中存在 NaN")

    median_similarity = torch.median(similarities_tensor).reshape(1)

    event_feature = torch.cat([absolute_diff, median_similarity])

    if torch.isnan(event_feature).any():
        raise ValueError("最终 event_feature 中存在 NaN")

    return event_feature

In [375]:
import numpy as np

In [377]:
def create_features(df):
    features = []
    valid_indices = []
    for index, row in df.iterrows():
        try: 
            feature = build_event_feature(row)
            features.append(feature.numpy())
            valid_indices.append(index)
        except Exception as error:
            print(f"跳过 video {row['ID']}: {error}")

    X = np.stack(features)
    valid_df = df.loc[valid_indices].copy()
    return X, valid_df

In [379]:
import pandas as pd

In [381]:
df_train = pd.read_excel("../data/labeled_data/train/train.xlsx")
df_test = pd.read_excel("../data/labeled_data/test/test.xlsx")

In [383]:
X_train, df_train_valid = create_features(df_train)

跳过 video C0449: [Errno 2] No such file or directory: '../data/labeled_data/img/C0449/after_1.png'
跳过 video C0409: [Errno 2] No such file or directory: '../data/labeled_data/img/C0409/after_1.png'


In [385]:
X_test, df_test_valid = create_features(df_test)

跳过 video C0173: [Errno 2] No such file or directory: '../data/labeled_data/img/C0173/after_1.png'


In [387]:
y_train = df_train_valid["consistency"].to_numpy()
y_test = df_test_valid["consistency"].to_numpy()

In [389]:
X_train.shape, X_test.shape

((281, 577), (70, 577))

In [391]:
from sklearn.linear_model import LogisticRegression

In [393]:
classifier = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

classifier.fit(X_train, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to

In [395]:
probabilities = classifier.predict_proba(X_test)
predictions = classifier.predict(X_test)

In [397]:
import numpy as np

In [399]:
def final_decision(prob, classes, confidence_threshold=0.7, gap_threshold=0.2):
    prob = np.asarray(prob, dtype=float)
    classes = np.asarray(classes)
    ranked_indices = np.argsort(prob)[::-1]
    top_index = ranked_indices[0]
    top_prob = prob[top_index]
    second_prob = prob[ranked_indices[1]]
    prob_gap = top_prob - second_prob
    if top_prob < confidence_threshold:
        return 2
    if prob_gap < gap_threshold:
        return 2
    return int(classes[top_index])

In [401]:
final_labels = [final_decision(prob, classes=classifier.classes_,confidence_threshold=0.7,gap_threshold=0.2) for prob in probabilities]

In [403]:
results = df_test_valid[["ID"]].reset_index(drop=True).copy()
results["true_label"] = np.asarray(y_test, dtype=int)
results["prediction"] = final_labels

In [405]:
for column_index, class_label in enumerate(classifier.classes_):
    results[f"prob_{int(class_label)}"] = probabilities[:, column_index]

In [407]:
sorted_probabilities = np.sort(probabilities, axis=1)

In [409]:
results["top_probability"] = sorted_probabilities[:, -1]
results["probability_gap"] = (sorted_probabilities[:, -1] - sorted_probabilities[:, -2])

In [411]:
confidence_threshold = 0.7
gap_threshold = 0.2
results["uncertain_reason"] = ["; ".join(
        reason
        for condition, reason in [
            (top < confidence_threshold, "最高概率不足"),
            (gap < gap_threshold, "前两类概率接近")
        ]
        if condition
    )
    for top, gap in zip(
        results["top_probability"],
        results["probability_gap"]
    )
]

In [413]:
correct_count = (results["true_label"] == results["prediction"]).sum()
uncertain_count = (results["prediction"] == 2).sum()

In [415]:
print("分类器类别：", classifier.classes_)
print("当前参与预测的测试样本数：", len(results))
print("预测正确数：", correct_count)
print(f"包含拒判的标签准确率：{correct_count / len(results):.2%}")
print("Uncertain 数量：", uncertain_count)
print("\n预测类别数量：")
print(results["prediction"].value_counts().sort_index())

display(results)

分类器类别： [0 1]
当前参与预测的测试样本数： 70
预测正确数： 41
包含拒判的标签准确率：58.57%
Uncertain 数量： 25

预测类别数量：
prediction
0    41
1     4
2    25
Name: count, dtype: int64


,ID,true_label,prediction,prob_0,prob_1,top_probability,probability_gap,uncertain_reason
0,C0495,0,1,0.236585,0.763415,0.763415,0.526830,
1,C0356,0,2,0.644562,0.355438,0.644562,0.289125,最高概率不足
2,C0471,0,0,0.981938,0.018062,0.981938,0.963876,
3,C0115,0,0,0.881445,0.118555,0.881445,0.762890,
4,C0482,0,0,0.968980,0.031020,0.968980,0.937960,
...,...,...,...,...,...,...,...,...
65,C0063,0,0,0.930605,0.069395,0.930605,0.861209,
66,C0465,0,2,0.657665,0.342335,0.657665,0.315329,最高概率不足
67,C0367,0,0,0.823260,0.176740,0.823260,0.646520,
68,C0216,0,0,0.774605,0.225395,0.774605,0.549210,


In [357]:
uncertain_count

np.int64(8)

In [359]:
correct_count

np.int64(52)

In [417]:
table = pd.crosstab(
    results["true_label"],
    results["prediction"]
).reindex(
    index=[0, 1],
    columns=[0, 1, 2],
    fill_value=0
)

table.index = ["真实 Pass", "真实 Fail"]
table.columns = ["预测 Pass", "预测 Fail", "Uncertain"]

display(table)

,预测 Pass,预测 Fail,Uncertain
真实 Pass,39,2,20
真实 Fail,2,2,5


In [419]:
display(
    results.loc[
        results["true_label"] == 1,
        [
            "ID",
            "true_label",
            "prediction",
            "prob_0",
            "prob_1",
            "top_probability",
            "probability_gap",
            "uncertain_reason"
        ]
    ].sort_values(["prediction", "prob_1"])
)

,ID,true_label,prediction,prob_0,prob_1,top_probability,probability_gap,uncertain_reason
44,C0327,1,0,0.820234,0.179766,0.820234,0.640467,
51,C0272,1,0,0.763928,0.236072,0.763928,0.527855,
42,C0106,1,1,0.286204,0.713796,0.713796,0.427593,
12,C0091,1,1,0.256471,0.743529,0.743529,0.487057,
55,C0273,1,2,0.636422,0.363578,0.636422,0.272843,最高概率不足
60,C0507,1,2,0.546185,0.453815,0.546185,0.092370,最高概率不足; 前两类概率接近
52,C0483,1,2,0.449590,0.550410,0.550410,0.100821,最高概率不足; 前两类概率接近
43,C0313,1,2,0.422017,0.577983,0.577983,0.155967,最高概率不足; 前两类概率接近
56,C0383,1,2,0.396293,0.603707,0.603707,0.207414,最高概率不足
